In [ ]:
# Prompt Processing Analysis for visit={{ params.visit }}

In [ ]:
visit_id = 2026030100037
butler_alias = "embargo"

In [ ]:
from lsst.daf.butler import Butler

In [ ]:
butler = Butler(butler_alias)

In [ ]:
# count_detectors_with_templates
from dataclasses import dataclass, field
import enum

import astropy.coordinates
import astropy.units as u
import lsst.geom
import lsst.sphgeom
import lsst.afw.cameraGeom
import lsst.obs.base

from lsst.daf.butler import Butler


@dataclass(frozen=True, kw_only=True)
class SimpleVisit:
    class CoordSys(enum.IntEnum):
        ICRS = 2

    class RotSys(enum.IntEnum):
        SKY = 2

    instrument: str
    filters: str
    coordinateSystem: CoordSys
    position: list[float] = field(compare=False)
    rotationSystem: RotSys
    cameraAngle: float
    detector: int

    def get_boresight_icrs(self):
        """Get ICRS coordinates of the boresight."""
        return astropy.coordinates.SkyCoord(*self.position, unit=u.degree, frame="icrs")

    def get_rotation_sky(self):
        """Get sky rotation angle."""
        return astropy.coordinates.Angle(self.cameraAngle, unit=u.degree)

    def predict_wcs(self, camera: lsst.afw.cameraGeom.Camera):
        """Calculate the expected detector WCS for this visit."""
        icrs = self.get_boresight_icrs()
        icrs = lsst.geom.SpherePoint(icrs.ra.degree, icrs.dec.degree, lsst.geom.degrees)

        rotation = self.get_rotation_sky()
        rotation = rotation.degree * lsst.geom.degrees

        detector = camera[self.detector]
        return lsst.obs.base.utils.createInitialSkyWcsFromBoresight(
            icrs, rotation, detector
        )

    def get_detector_icrs_region(self, camera: lsst.afw.cameraGeom.Camera):
        """Return the detector region in ICRS coordinates."""
        wcs = self.predict_wcs(camera)
        detector = camera[self.detector]
        corners = wcs.pixelToSky(detector.getCorners(lsst.afw.cameraGeom.PIXELS))
        return lsst.sphgeom.ConvexPolygon.convexHull([c.getVector() for c in corners])


def overlap_templates(butler, visit_id, detector_id, exp_record=None):
    """Check if templates overlap with the detector footprint for a given visit.

    Parameters
    ----------
    butler : `lsst.daf.butler.Butler`
        Butler instance to query.
    visit_id : `int`
        Visit ID to check.
    detector_id : `int`
        Detector ID to check.
    exp_record : exposure record, optional
        Pre-fetched exposure record. If None, will query it.

    Returns
    -------
    has_overlap : `bool`
        True if overlapping templates are found, False otherwise.
    """
    # Get exposure record if not provided
    if exp_record is None:
        try:
            exp_record = butler.query_dimension_records(
                "exposure", instrument="LSSTCam", visit=visit_id
            )[0]
        except Exception:
            return False

    visit = SimpleVisit(
        instrument=exp_record.instrument,
        detector=detector_id,
        filters=exp_record.physical_filter,
        coordinateSystem=SimpleVisit.CoordSys.ICRS,
        position=[exp_record.tracking_ra, exp_record.tracking_dec],
        rotationSystem=SimpleVisit.RotSys.SKY,
        cameraAngle=exp_record.sky_angle,
    )

    camera = butler.get("camera", instrument="LSSTCam", collections="LSSTCam/calib")
    region = visit.get_detector_icrs_region(camera)

    try:
        templates = butler.query_datasets(
            "template_coadd",
            collections="LSSTCam/templates",
            data_id={
                "instrument": visit.instrument,
                "skymap": "lsst_cells_v2",
                "physical_filter": visit.filters,
            },
            where="patch.region OVERLAPS search_region",
            bind={"search_region": region},
            find_first=True,
            explain=False,
            limit=1,
        )
        # Check if we got any results
        return len(templates) > 0
    except Exception:
        return False


def count_detectors_with_templates(butler, visit_id):
    """Count how many detectors have overlapping templates for a given visit.

    Parameters
    ----------
    butler : `lsst.daf.butler.Butler`
        Butler instance to query.
    visit_id : `int`
        Visit ID to check.

    Returns
    -------
    count : `int`
        Number of detectors (0-188, excluding off detectors) that have overlapping templates.
    """
    off_detectors = (
        120,
        122,
        0,
        20,
        27,
        65,
        123,
        161,
        168,
        188,
        1,
        19,
        30,
        68,
        158,
        169,
        187,
    )

    # Query exposure record once for all detectors
    try:
        exp_record = butler.query_dimension_records(
            "exposure", instrument="LSSTCam", visit=visit_id
        )[0]
    except Exception:
        return 0

    count = 0
    for detector_id in range(189):
        if detector_id in off_detectors:
            continue
        if overlap_templates(butler, visit_id, detector_id, exp_record=exp_record):
            count += 1
    return count

In [ ]:
num_detectors = count_detectors_with_templates(butler, visit_id)
print(f"{num_detectors} detectors have overlapping templates for {visit_id}")

In [ ]:
# get_detector_timelines
import json
import re
import requests
from datetime import datetime, timezone
from lsst.daf.butler import Butler, EmptyQueryResultError


# Configurable search phrases
START_PHRASE = "Unpacked message as"
FINISH_PHRASE = "Request took"
FINISH_PHRASE = "Main pipeline successfully run"
MIDDLE_PHRASES = {
    "ready": "Waiting for snap",
    "ingest": "Ingesting image",
    # Add more checkpoints here as needed:
    # "calibrate": "Calibrating detector",
    # "diff_image": "Generating difference image",
}

# Set which checkpoint to use as time reference
# Options: None, "start", "finish", "exposure_end", or any key from MIDDLE_PHRASES
TIME_REFERENCE = "exposure_end"  # Set to None or "start" to use start time as reference

# Pipelines to exclude from counting
EXCLUDED_PIPELINES = {"Preprocessing"}

# Pattern to extract duration from "Request took X.XXX s." messages
DURATION_PATTERN = r"Request took\s+([\d.]+)\s+s\."
# Pattern to extract result from "Result: Success" messages
RESULT_PATTERN = r"Result:\s+(\w+)"
# Pattern to extract pipeline name from "Running '/app/pipelines/LSSTCam/ApPipe.yaml'" messages
PIPELINE_PATTERN = r"Running\s+'/app/pipelines/LSSTCam/(?!Preprocessing)(\w+)\.yaml'"


def loki_query(
    group_id,
    time_range,
    namespace="vcluster--usdf-prompt-processing",
    container="lsstcam",
):
    """Query Grafana Loki for log records matching a groupId.

    Parameters
    ----------
    group_id : `str`
        The groupId to filter on.
    time_range : `tuple` of `float`, optional
        (start_timestamp, end_timestamp) in Unix seconds.
    namespace : `str`, optional
        Kubernetes namespace for the logs.
    container : `str`, optional
        Container name for the logs.

    Returns
    -------
    records : `list` of `dict`
        List of log records with 'message', 'group', 'detector', 'timestamp' keys.
    """
    loki_url = "http://sdfloki.slac.stanford.edu:80"

    # Build LogQL query - line filter BEFORE json parsing
    # Combine all search phrases with OR, be more specific with Running pattern
    all_phrases = [START_PHRASE, FINISH_PHRASE, "Running '/app/pipelines"] + list(
        MIDDLE_PHRASES.values()
    )
    search_pattern = "|".join(all_phrases)
    logql = (
        f'{{namespace="{namespace}", container="{container}"}} |~ "{search_pattern}"'
    )

    params = {
        "query": logql,
        "limit": 50000,  # Adjust if you expect more than 10k log lines
    }

    if time_range:
        params["start"] = int(time_range[0] * 1e9)  # Convert to nanoseconds
        params["end"] = int(time_range[1] * 1e9)

    try:
        response = requests.get(
            f"{loki_url}/loki/api/v1/query_range", params=params, timeout=60
        )
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        raise RuntimeError(f"Loki query failed: {e}")

    records = []
    if data.get("status") == "success":
        for stream in data.get("data", {}).get("result", []):
            for value in stream.get("values", []):
                timestamp_ns, log_line = value

                try:
                    # Parse JSON log line
                    log_json = json.loads(log_line)
                    message = log_json.get("message", "")

                    # Extract group and detector from message context
                    group = log_json.get("group")
                    detector = None

                    # Try to get detector from structured field
                    detector_field = log_json.get("detector")
                    if detector_field is not None:
                        try:
                            detector = int(detector_field)
                        except (ValueError, TypeError):
                            pass

                    # If not in structured fields, try to parse from message
                    if detector is None and "detector=" in message:
                        try:
                            detector_str = (
                                message.split("detector=")[1]
                                .split(",")[0]
                                .split(")")[0]
                            )
                            detector = int(detector_str)
                        except (IndexError, ValueError):
                            pass

                    # Only include messages for the requested group_id
                    if group == group_id or group_id in message:
                        records.append(
                            {
                                "message": message,
                                "group": group,
                                "detector": detector,  # Always int or None
                                "timestamp": int(timestamp_ns) / 1e9,
                                "asctime": log_json.get("asctime"),
                            }
                        )

                except json.JSONDecodeError:
                    # Skip non-JSON lines
                    continue

    return records

def _parse_asctime(asctime_str):
    """Parse asctime string to timestamp.

    Parameters
    ----------
    asctime_str : `str`
        Timestamp string in format "2026-03-16T19:16:11.217+0000"

    Returns
    -------
    timestamp : `float` or `None`
        Unix timestamp in seconds, or None if parsing failed.
    """
    if not asctime_str:
        return None
    try:
        # Parse ISO format with timezone
        dt = datetime.fromisoformat(asctime_str.replace("+0000", "+00:00"))
        return dt.timestamp()
    except (ValueError, AttributeError):
        return None
        
def get_detector_timelines(
    butler, visit_id, namespace="vcluster--usdf-prompt-processing", container="lsstcam"
):
    """Get per-detector timeline data for plotting.

    Parameters
    ----------
    butler : `lsst.daf.butler.Butler`
        Butler instance to query for exposure records.
    visit_id : `int`
        Visit ID to check.
    namespace : `str`, optional
        Kubernetes namespace for the logs.
    container : `str`, optional
        Container name for the logs.

    Returns
    -------
    timeline_data : `dict`
        Dictionary with keys:
        - 'detectors': List of detector IDs that have data
        - 'reference_time': The reference timestamp used
        - 'time_reference_name': Name of the reference (e.g., "exposure_end")
        - 'start_times': Dict mapping detector -> relative start time
        - 'finish_times': Dict mapping detector -> relative finish time
        - 'checkpoint_times': Dict mapping checkpoint_name -> dict(detector -> relative time)
        - 'durations': Dict mapping detector -> actual duration (start to finish)
        - 'pipelines': Dict mapping detector -> pipeline name
    """
    # Get exposure record
    try:
        exp_record = list(
            butler.query_dimension_records(
                "exposure", instrument="LSSTCam", visit=visit_id
            )
        )[0]
    except EmptyQueryResultError:
        return {
            "detectors": [],
            "reference_time": None,
            "time_reference_name": TIME_REFERENCE if TIME_REFERENCE else "start",
            "start_times": {},
            "finish_times": {},
            "checkpoint_times": {},
            "durations": {},
            "pipelines": {},
        }

    group_id = exp_record.group

    # Get exposure end time if needed
    exposure_end_timestamp = None
    if exp_record.timespan and exp_record.timespan.end:
        exposure_end_dt = exp_record.timespan.end.utc.datetime
        if exposure_end_dt.tzinfo is None:
            exposure_end_dt = exposure_end_dt.replace(tzinfo=timezone.utc)
        exposure_end_timestamp = exposure_end_dt.timestamp()

    # Define time range for log query
    if exp_record.timespan and exp_record.timespan.begin:
        start_time = exp_record.timespan.begin.tai.unix - 3600
        end_time = start_time + 7200
        time_range = (start_time, end_time)
    else:
        time_range = None

    # Query logs
    try:
        log_records = loki_query(group_id, time_range, namespace, container)
    except Exception as e:
        print(f"Log query error: {e}")
        return {
            "detectors": [],
            "reference_time": None,
            "time_reference_name": TIME_REFERENCE if TIME_REFERENCE else "start",
            "start_times": {},
            "finish_times": {},
            "checkpoint_times": {},
            "durations": {},
            "pipelines": {},
        }

    # Parse log records
    detector_start_times = {}  # detector -> asctime string
    detector_finish_times = {}
    detector_checkpoint_times = {name: {} for name in MIDDLE_PHRASES.keys()}
    detector_pipelines = {}  # detector -> pipeline name

    for record in log_records:
        message = record.get("message", "")
        detector = record.get("detector")
        asctime = record.get("asctime")

        if START_PHRASE in message:
            if detector is not None and asctime:
                detector_start_times[detector] = asctime

        for checkpoint_name, checkpoint_phrase in MIDDLE_PHRASES.items():
            if checkpoint_phrase in message:
                if detector is not None and asctime:
                    detector_checkpoint_times[checkpoint_name][detector] = asctime

        pipeline_match = re.search(PIPELINE_PATTERN, message)
        if pipeline_match:
            pipeline_name = pipeline_match.group(1)
            if pipeline_name not in EXCLUDED_PIPELINES and detector is not None:
                detector_pipelines[detector] = pipeline_name

        if FINISH_PHRASE in message:
            if detector is not None and asctime:
                detector_finish_times[detector] = asctime

    # Determine reference time
    reference = TIME_REFERENCE if TIME_REFERENCE else "start"

    if reference == "exposure_end":
        if exposure_end_timestamp is None:
            reference = "start"
            reference_timestamp = None
            detector_reference_times = detector_start_times
        else:
            reference_timestamp = exposure_end_timestamp
            detector_reference_times = None
    elif reference in ("start", None):
        reference_timestamp = None
        detector_reference_times = detector_start_times
        reference = "start"
    elif reference == "finish":
        reference_timestamp = None
        detector_reference_times = detector_finish_times
    elif reference in MIDDLE_PHRASES.keys():
        reference_timestamp = None
        detector_reference_times = detector_checkpoint_times[reference]
    else:
        reference_timestamp = None
        detector_reference_times = detector_start_times
        reference = "start"

    # Helper function
    def get_reference_time(detector):
        if reference_timestamp is not None:
            return reference_timestamp
        elif detector_reference_times and detector in detector_reference_times:
            return _parse_asctime(detector_reference_times[detector])
        return None

    # Calculate relative times and durations
    start_times_relative = {}
    finish_times_relative = {}
    checkpoint_times_relative = {name: {} for name in MIDDLE_PHRASES.keys()}
    durations = {}

    # Get all detectors
    all_detectors = set(detector_start_times.keys()) | set(detector_finish_times.keys())
    for checkpoint_times in detector_checkpoint_times.values():
        all_detectors |= set(checkpoint_times.keys())

    for detector in all_detectors:
        ref_ts = (
            get_reference_time(detector)
            if reference_timestamp is None
            else reference_timestamp
        )

        if detector in detector_start_times:
            start_ts = _parse_asctime(detector_start_times[detector])
            if start_ts is not None and ref_ts is not None:
                start_times_relative[detector] = start_ts - ref_ts

        if detector in detector_finish_times:
            finish_ts = _parse_asctime(detector_finish_times[detector])
            if finish_ts is not None and ref_ts is not None:
                finish_times_relative[detector] = finish_ts - ref_ts

            # Calculate actual duration
            if detector in detector_start_times:
                start_ts = _parse_asctime(detector_start_times[detector])
                if start_ts is not None and finish_ts is not None:
                    durations[detector] = finish_ts - start_ts

        for checkpoint_name in MIDDLE_PHRASES.keys():
            if detector in detector_checkpoint_times[checkpoint_name]:
                checkpoint_ts = _parse_asctime(
                    detector_checkpoint_times[checkpoint_name][detector]
                )
                if checkpoint_ts is not None and ref_ts is not None:
                    checkpoint_times_relative[checkpoint_name][detector] = (
                        checkpoint_ts - ref_ts
                    )

    return {
        "detectors": sorted(list(all_detectors)),
        "reference_time": (
            reference_timestamp if reference_timestamp is not None else 0.0
        ),
        "time_reference_name": reference,
        "start_times": start_times_relative,
        "finish_times": finish_times_relative,
        "checkpoint_times": checkpoint_times_relative,
        "durations": durations,
        "pipelines": detector_pipelines,
    }



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

timeline = get_detector_timelines(butler, visit_id)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Collect all checkpoint data
checkpoint_data = {}

# Start times
if timeline['start_times']:
    checkpoint_data['Start'] = list(timeline['start_times'].values())

# Middle checkpoints
for checkpoint_name in ['ready', 'ingest']:
    if checkpoint_name in timeline['checkpoint_times'] and timeline['checkpoint_times'][checkpoint_name]:
        checkpoint_data[checkpoint_name.capitalize()] = list(timeline['checkpoint_times'][checkpoint_name].values())

# Finish times
if timeline['finish_times']:
    checkpoint_data['Finish'] = list(timeline['finish_times'].values())

# Determine common bin range across all checkpoints
all_times = []
for times in checkpoint_data.values():
    all_times.extend(times)

if all_times:
    min_time = min(all_times)
    max_time = max(all_times)
    bins = np.linspace(min_time, max_time, 50)  # Common bins for all checkpoints
    
    # Plot histograms for each checkpoint with common bins
    colors = ['blue', 'green', 'orange', 'red', 'purple']
    for i, (label, times) in enumerate(checkpoint_data.items()):
        ax.hist(times, bins=bins, alpha=0.5, label=label, color=colors[i % len(colors)], edgecolor='black')
    
    # Mark the reference time (x=0)
    ax.axvline(0, color='black', linestyle='--', linewidth=2, label=f'Reference: {timeline["time_reference_name"]}')
    
    ax.set_xlabel(f'Time relative to {timeline["time_reference_name"]} (seconds)')
    ax.set_ylabel('Number of detectors')
    ax.set_title(f'Visit {visit_id}: Checkpoint Time Distributions')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print statistics for each checkpoint
    print(f"\n=== Checkpoint Statistics (relative to {timeline['time_reference_name']}) ===\n")
    for label, times in checkpoint_data.items():
        print(f"{label} Count: {len(times)} Min: {min(times):.2f}s Max: {max(times):.2f}s Mean: {np.mean(times):.2f}s Median:{np.median(times):.2f}s")
else:
    print("No checkpoint timing data available")